# **1. Perkenalan Dataset**

## 1. Sumber dan Deskripsi Dataset

Dataset yang digunakan dalam proyek ini adalah **Student Performance Analytics Dataset** yang diperoleh dari platform Kaggle. Dataset ini berisi lebih dari 10.000 data siswa dengan berbagai atribut yang merepresentasikan performa akademik, kebiasaan belajar, serta faktor latar belakang yang berpotensi memengaruhi hasil belajar.

Setiap baris data merepresentasikan satu individu siswa dengan sejumlah fitur yang dapat dikelompokkan sebagai berikut:

* **Academic Performance**: nilai assignment, midterm, final exam, dan partisipasi
* **Study Behavior**: durasi belajar harian dan waktu tidur
* **Engagement**: tingkat kehadiran dan partisipasi kelas
* **Background Factors**: pendidikan orang tua, akses internet, dan kelas tambahan

Dataset ini juga menyediakan variabel target yang dapat digunakan dalam pemodelan machine learning, yaitu:

* **Overall Score** (numerik → regresi)
* **Grade (A–F)** (kategori → klasifikasi)

Dataset ini memiliki struktur yang bersih tanpa missing values sehingga dapat langsung digunakan dalam proses pemodelan tanpa tahap imputasi yang kompleks.

---

## 2. Tujuan Penggunaan Dataset

Dataset ini digunakan untuk membangun model machine learning yang bertujuan:

* Mengklasifikasikan performa siswa berdasarkan kategori nilai (Grade)
* Mengidentifikasi faktor-faktor yang paling berpengaruh terhadap performa akademik
* Menguji performa model melalui eksperimen yang terdokumentasi menggunakan MLflow

Pemilihan dataset ini didasarkan pada kompleksitas fitur yang cukup representatif untuk studi kasus machine learning, serta kemudahan dalam integrasi ke dalam pipeline MLOps.


# **2. Import Library**

## 2. Import Library

Pada tahap ini dilakukan proses impor pustaka Python yang diperlukan untuk mendukung seluruh pipeline machine learning, mulai dari pengolahan data, pemodelan, hingga pencatatan eksperimen menggunakan MLflow.

Library yang digunakan meliputi:

* **pandas** → digunakan untuk manipulasi dan analisis data berbasis tabel
* **numpy** → digunakan untuk operasi numerik dan pembuatan range parameter
* **scikit-learn** → digunakan untuk proses pemodelan, pembagian data (train-test split), serta evaluasi model
* **mlflow** → digunakan untuk melakukan tracking eksperimen, pencatatan parameter, metrik, dan model

Berikut adalah library yang diimpor:



Import library ini menjadi fondasi dalam membangun pipeline machine learning yang terstruktur dan dapat direproduksi, terutama dalam konteks penerapan MLOps.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! pip install mlflow

In [ ]:
import pandas as pd
import numpy as np

# visualisasi
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn

# **3. Memuat Dataset**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/KERJAAN/MACHINE LEARNING/SMSML DIcoding/student_performance_data.csv')

In [ ]:
# menampilkan semua kolom
pd.set_option('display.max_columns', None)
# melihat sample data
df.head()

In [ ]:
# melihat informasi dataset
df.info()

In [ ]:
# melihat missing value
df.isnull().sum()

In [ ]:
# melihat duplikat data
df.duplicated().sum()

In [ ]:
# statistik deskriptif
df.describe()

In [ ]:
# melihat nilai untuk kolom gender
df['gender'].value_counts()

In [ ]:
# melihat nilai untuk kolom internet_access
df['internet_access'].value_counts()

In [ ]:
# melihat nilai untuk kolom extra_classes
df['extra_classes'].value_counts()

In [ ]:
# melihat nilai untuk kolom parent_education
df['parent_education'].value_counts()

In [ ]:
# melihat nilai untuk kolom grade
df['grade'].value_counts()  

### Visualisasi

In [ ]:
gender_counts = df["gender"].value_counts()

# plot pie chart
plt.figure(figsize=(6,6))
plt.pie(
    gender_counts,
    labels=gender_counts.index,
    autopct='%1.1f%%',
    startangle=90
)

plt.title("Distribusi Gender")
plt.axis('equal') 
plt.show()

In [ ]:
df["study_hours_category"] = pd.cut(
    df["study_hours_per_day"],
    bins=[1, 3, 5, 7, 9, 10],
    labels=["very_low", "low", "medium", "high", "very_high"],
    include_lowest=True
)

In [ ]:
df["study_hours_category"].value_counts()

In [ ]:

palette_grade = {
    "A": "#2ecc71",
    "B": "#3498db",
    "C": "#f1c40f",
    "D": "#e67e22",
    "F": "#e74c3c"
}

fig, axes = plt.subplots(4, 3, figsize=(24, 20))
fig.suptitle('Feature vs Grade Analysis', fontsize=20)

features = [
    'gender',
    'study_hours_per_day',
    'attendance_percentage',
    'assignment_score',
    'midterm_score',
    'final_exam_score',
    'participation_score',
    'internet_access',
    'extra_classes',
    'parent_education',
    'sleep_hours',
    'overall_score'
]

for ax, col in zip(axes.flatten(), features):

    if df[col].dtype == "object":
        sns.countplot(data=df, x=col, hue='grade', palette=palette_grade, ax=ax)
    else:
        sns.barplot(data=df, x='grade', y=col, palette=palette_grade, ax=ax)

    ax.set_title(f"{col} vs grade")
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
df = df.drop(columns=["overall_score", "student_id"])

In [ ]:
numeric = df.select_dtypes(include=["int64", "float64"]).columns
categoric = df.select_dtypes(include=["object"]).columns
# melihat nilai untuk tiap kategori baik numerik maupun kategorik

print("jumlah kolom numerik", len(numeric))
print("jumlah kolom kategorik", len(categoric))

In [ ]:
import numpy as np
import pandas as pd

print(f'Jumlah baris: {len(df)}')

outlier = []
low_lim = []
high_lim = []

for col in numeric:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    low_limit = Q1 - 1.5 * IQR
    high_limit = Q3 + 1.5 * IQR

    outlier_count = ((df[col] < low_limit) | (df[col] > high_limit)).sum()

    outlier.append(outlier_count)
    low_lim.append(low_limit)
    high_lim.append(high_limit)

result = pd.DataFrame({
    "Column": numeric,
    "Lower Limit": low_lim,
    "Upper Limit": high_lim,
    "Outlier Count": outlier
})

result

In [ ]:
mask = np.zeros(len(df), dtype=bool)

for col in numeric:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR

    mask |= (df[col] < low) | (df[col] > high)

print("Total baris mengandung outlier:", mask.sum())

In [ ]:
# Cek Outlier
print(f'Jumlah baris: {len(df)}')

outlier = []
no_outlier = []
is_outlier = []
low_lim = []
high_lim = []

filtered_entries = np.array([True] * len(df))
for col in numeric:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    low_limit = Q1 - (IQR * 1.5)
    high_limit = Q3 + (IQR * 1.5)

    # mulai filter outlier
    filter_outlier = ((df[col] >= low_limit) & (df[col] <= high_limit))
    outlier.append(len(df[~filter_outlier]))
    no_outlier.append(len(df[filter_outlier]))
    is_outlier.append(df[col][~filter_outlier].any())
    low_lim.append(low_limit)
    high_lim.append(high_limit)

    filtered_entries = ((df[col] >= low_limit) & (df[col] <= high_limit)) & filtered_entries

print("Outlier All Data :", len(df[~filtered_entries]))
print("Not Outlier All Data :", len(df[filtered_entries]))
print()

# Pastikan numeric diubah ke list jika masih berupa Index
pd.DataFrame({
    "Column Name": list(numeric),
    "is Outlier": is_outlier,
    "Lower Limit": low_lim,
    "Upper Limit": high_lim,
    "Outlier": outlier,
    "No Outlier": no_outlier
})

### Analisis Outlier

Berdasarkan metode Interquartile Range (IQR), tidak ditemukan outlier pada seluruh fitur numerik dalam dataset. Hal ini ditunjukkan oleh jumlah data yang berada di luar batas bawah dan atas (lower dan upper limit) yang bernilai nol untuk setiap fitur.

Kondisi ini menunjukkan bahwa dataset memiliki distribusi yang relatif stabil dan tidak mengandung nilai ekstrem yang signifikan. Oleh karena itu, tidak dilakukan proses penghapusan outlier agar informasi data tetap terjaga.
